In [1]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq

### **Sampling from the estimated CDF**

$$
U_j \sim \mathrm{Uniform}(0,1),
\qquad j = 1,\ldots,m
$$

$$
X_j^* = F_n^{-1}(U_j)
$$



In [2]:
def sample_from_cdf(x_grid, F, n, seed=None):

    rng = np.random.default_rng(seed)
    u = rng.uniform(0, 1, size=n)
    
    samples = np.interp(u, F, x_grid)

    return samples

### **Sampling from the R-BP distribution**

$P_n$ : the estimated R-BP distribution after observing $x_1,\ldots,x_n$.

$$
P_n(x) = (1-\alpha_n)P_{n-1}(x)+\alpha_n H_\rho\left\{P_{n-1}(x),P_{n-1}(x_n)\right\}
$$

Let

$$
Y \sim P_n
$$

Then

$$
P_n(Y)=U_n,
\qquad
U_n \sim \mathrm{Uniform}(0,1)
$$

Let

$$
U_i = P_i(Y),
\qquad
v_i = P_{i-1}(x_i)
$$

Then

$$
U_n = (1-\alpha_n)U_{n-1}+\alpha_n H_\rho(U_{n-1},v_n)
$$

$$
U_{n-1} = (1-\alpha_{n-1})U_{n-2}+\alpha_{n-1}H_\rho(U_{n-2},v_{n-1})
$$

$$
\vdots
$$

$$
U_1 = (1-\alpha_1)U_0+\alpha_1H_\rho(U_0,v_1)
$$

Equivalently,

$$
(1-\alpha_i)U_{i-1}+\alpha_iH_\rho(U_{i-1},v_i)-U_i=0
$$

$$
f(U_{i-1}) = (1-\alpha_i)U_{i-1}+\alpha_iH_\rho(U_{i-1},v_i)-U_i
$$

Root finding 

$$
f(U_{i-1}) = 0
$$

Finally,

$$
Y = P_0^{-1}(U_0)
$$

In [3]:
def R_BP_sample(x, x_grid, rho, n, P0_grid, P0_inv, seed=None):
    rng = np.random.default_rng(seed)

    # P_0
    P = P0_grid.copy()

    v_list = []
    alpha_list = []

    for i in range(len(x)):

        alpha = 1 / (i + 2)
        v_i = np.interp(x[i], x_grid, P)
        H_rho = gaussian_conditional_copula_cdf(P, v_i, rho)

        P = (1 - alpha) * P + alpha * H_rho

        v_list.append(v_i)
        alpha_list.append(alpha)

    samples = []

    for _ in range(n):

        # U_n
        U = rng.uniform(0, 1)

        # backward recursion
        for i in reversed(range(len(x))):

            alpha = alpha_list[i]
            v_i = v_list[i]

            def eq(U_prev):

                f = ((1 - alpha) * U_prev + alpha * gaussian_conditional_copula_cdf(U_prev, v_i, rho) - U)
                
                return f

            # finding root : f(U_pre) = 0 
            U_prev = brentq(eq, 1e-10, 1 - 1e-10)

            U = U_prev

        # Y = P_0^{-1}(U_0)
        y = P0_inv(U)

        samples.append(y)


    return np.array(samples)